In [87]:
import pandas as pd
import sys                 # Permet de modifier les chemins de recherche de Python
from pathlib import Path   # Manipulation des chemins
PROJECT_ROOT = Path.cwd().parent   # C:\RetailVision
sys.path.insert(0, str(PROJECT_ROOT))
# Ajoute C:\RetailVision aux dossiers où Python cherche les modules

from src.etl.extract import extract_data
from src.etl.transform import (
    check_missing_value,
    check_duplicates_rows,
    check_duplicates_keys,
    check_data_types,
    data_quality_report,
    normalize_text_columns,
    convert_datetime_columns,
    clean_orders,
    clean_products,
    clean_order_payments,
    clean_order_reviews,
    clean_geolocation

)
from src.etl.load import save_to_silver

In [2]:
datasets=extract_data()

In [3]:
primary_keys = {
    "olist_orders_dataset": "order_id",
    "olist_customers_dataset": "customer_id",
    "olist_products_dataset": "product_id",
    "olist_sellers_dataset": "seller_id",
    "olist_order_reviews_dataset": "review_id",
    "olist_order_payments_dataset": "order_id",
    "olist_order_items_dataset": "order_id",
    "olist_geolocation_dataset": None,
    "product_category_name_translation": "product_category_name"
}

In [4]:
# Pour Chaque Table
for table_name, df in datasets.items():

    pk = primary_keys[table_name]

    report = data_quality_report(df, pk)

    
    print(f"Table : {table_name}")
    print("=" * 40)
    print("")

    for key, value in report.items():
        print(f"{key}: {value}")
    
    print(end="\n"*2)  
  

Table : olist_customers_dataset

Rows: 99441
Columns: 5
Missing Values Count: 0
Duplicate Rows: 0
Duplicate Keys: 0


Table : olist_geolocation_dataset

Rows: 1000163
Columns: 5
Missing Values Count: 0
Duplicate Rows: 261831
Duplicate Keys: None


Table : olist_orders_dataset

Rows: 99441
Columns: 8
Missing Values Count: 4908
Duplicate Rows: 0
Duplicate Keys: 0


Table : olist_order_items_dataset

Rows: 112650
Columns: 7
Missing Values Count: 0
Duplicate Rows: 0
Duplicate Keys: 13984


Table : olist_order_payments_dataset

Rows: 103886
Columns: 5
Missing Values Count: 0
Duplicate Rows: 0
Duplicate Keys: 4446


Table : olist_order_reviews_dataset

Rows: 99224
Columns: 7
Missing Values Count: 145903
Duplicate Rows: 0
Duplicate Keys: 814


Table : olist_products_dataset

Rows: 32951
Columns: 9
Missing Values Count: 2448
Duplicate Rows: 0
Duplicate Keys: 0


Table : olist_sellers_dataset

Rows: 3095
Columns: 4
Missing Values Count: 0
Duplicate Rows: 0
Duplicate Keys: 0


Table : product_ca

In [5]:
# Pour Toutes Les Tables En Meme Temps
reports = []
for table_name, df in datasets.items() :

    pk = primary_keys[table_name]
    report = data_quality_report(df,pk)
    report['Table'] = table_name
    reports.append(report)

summary = pd.DataFrame(reports)
print(summary)


      Rows  Columns  Missing Values Count  Duplicate Rows  Duplicate Keys  \
0    99441        5                     0               0             0.0   
1  1000163        5                     0          261831             NaN   
2    99441        8                  4908               0             0.0   
3   112650        7                     0               0         13984.0   
4   103886        5                     0               0          4446.0   
5    99224        7                145903               0           814.0   
6    32951        9                  2448               0             0.0   
7     3095        4                     0               0             0.0   
8       71        2                     0               0             0.0   

                               Table  
0            olist_customers_dataset  
1          olist_geolocation_dataset  
2               olist_orders_dataset  
3          olist_order_items_dataset  
4       olist_order_payments_datas

In [6]:
list(datasets.keys())

['olist_customers_dataset',
 'olist_geolocation_dataset',
 'olist_orders_dataset',
 'olist_order_items_dataset',
 'olist_order_payments_dataset',
 'olist_order_reviews_dataset',
 'olist_products_dataset',
 'olist_sellers_dataset',
 'product_category_name_translation']

In [7]:
def Affichage_Dic(data):
    for key, value in data.items():
        print(f"- {key}: {value}")

In [8]:
customers  = datasets["olist_customers_dataset"]
reportCustomers=data_quality_report(customers ,primary_keys["olist_customers_dataset"])
print(" Customers Report : \n")
Affichage_Dic(reportCustomers)

 Customers Report : 

- Rows: 99441
- Columns: 5
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [9]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [10]:
customers_clean = normalize_text_columns(
    customers,
    ["customer_city", "customer_state"]
)

In [11]:
Affichage_Dic(data_quality_report(customers_clean, "customer_id"))

- Rows: 99441
- Columns: 5
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [12]:
#Sauvegarder dans Silver
save_to_silver(customers,"olist_customers_dataset")

olist_customers_dataset : Saved


In [13]:
orders = datasets["olist_orders_dataset"]
orders.dtypes 

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [14]:
#Si on remarque probleme au niveau du type des dates
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
] #les collonnes qui contienent des Dates

orders = convert_datetime_columns(orders, date_columns)
orders.dtypes 


order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [15]:
reportOrders = data_quality_report(orders, primary_keys["olist_orders_dataset"])
print(" Orders Report : \n")
Affichage_Dic(reportOrders)

 Orders Report : 

- Rows: 99441
- Columns: 8
- Missing Values Count: 4908
- Duplicate Rows: 0
- Duplicate Keys: 0


In [16]:
print("Missing Values pour Orders:")
check_missing_value(orders)

Missing Values pour Orders:


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [17]:
# nombre de commandes par statut pour les commandes où la date de livraison est manquante
orders[ orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [18]:
# Vérifier les lignes de dates manquantes
orders[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [19]:
orders[orders["order_delivered_carrier_date"].isnull()]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [20]:
# Vérifier les lignes de dates manquantes
orders[(orders["order_status"] == "delivered") & (orders["order_delivered_carrier_date"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


In [21]:
orders[orders["order_approved_at"].isnull()]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [22]:
# Vérifier les lignes de dates manquantes
orders[(orders["order_status"] == "delivered") & (orders["order_approved_at"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


In [23]:
orders.groupby("order_status").count()

,order_id,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,,,
approved,2,2,2,2,0,0,2
canceled,625,625,625,484,75,6,625
created,5,5,5,0,0,0,5
delivered,96478,96478,96478,96464,96476,96470,96478
invoiced,314,314,314,314,0,0,314
processing,301,301,301,301,0,0,301
shipped,1107,1107,1107,1107,1107,0,1107
unavailable,609,609,609,609,0,0,609


In [24]:
# orders.groupby("order_status")["nom_colonne"].count() -> si tu veux une colonne specifique
orders.groupby("order_status")["order_delivered_customer_date"].count() 

order_status
approved           0
canceled           6
created            0
delivered      96470
invoiced           0
processing         0
shipped            0
unavailable        0
Name: order_delivered_customer_date, dtype: int64

In [25]:
for column in date_columns :
    print(f'Pour column : {column} :')
    print(orders[orders[column].isnull()].groupby("order_status")[column].size())
    print()

Pour column : order_purchase_timestamp :
Series([], Name: order_purchase_timestamp, dtype: int64)

Pour column : order_approved_at :
order_status
canceled     141
created        5
delivered     14
Name: order_approved_at, dtype: int64

Pour column : order_delivered_carrier_date :
order_status
approved         2
canceled       550
created          5
delivered        2
invoiced       314
processing     301
unavailable    609
Name: order_delivered_carrier_date, dtype: int64

Pour column : order_delivered_customer_date :
order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_delivered_customer_date, dtype: int64

Pour column : order_estimated_delivery_date :
Series([], Name: order_estimated_delivery_date, dtype: int64)



In [26]:
# DataFrame.groupby("colonne_de_regroupement")["colonne_à_analyser"]
#orders.groupby("order_status")["order_id"].size()

In [27]:
orders_clean = clean_orders(orders)
print(orders.shape)
print(orders_clean.shape)

(99441, 8)
(99418, 8)


In [28]:
orders_clean = normalize_text_columns(
    orders_clean,
    ["order_status"]
)

In [29]:
Affichage_Dic(data_quality_report(orders_clean, "order_id"))

- Rows: 99418
- Columns: 8
- Missing Values Count: 4884
- Duplicate Rows: 0
- Duplicate Keys: 0


In [30]:
save_to_silver(orders_clean,"olist_orders_dataset")

olist_orders_dataset : Saved


In [31]:
products = datasets["olist_products_dataset"]
reportProducts = data_quality_report(products,"product_id")
print("Pour Table Product :")
print(f'{Affichage_Dic(reportProducts)}')

Pour Table Product :
- Rows: 32951
- Columns: 9
- Missing Values Count: 2448
- Duplicate Rows: 0
- Duplicate Keys: 0
None


In [32]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [33]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [34]:
products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [35]:
products[products["product_category_name"].isna()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [36]:
products["product_category_name"].isna().sum()

np.int64(610)

In [37]:
products_clean = clean_products(products)

In [38]:
# Vérifier la différence après le nettoyage
print(f"Avant : {products['product_category_name'].isna().sum()}")
print(f"Apres : {products_clean['product_category_name'].isna().sum()}")

Avant : 610
Apres : 0


In [39]:
products_clean = normalize_text_columns(
    products_clean,
    ["product_category_name"]
)

In [40]:
Affichage_Dic(data_quality_report(products_clean, "product_id"))

- Rows: 32951
- Columns: 9
- Missing Values Count: 1838
- Duplicate Rows: 0
- Duplicate Keys: 0


In [41]:
save_to_silver(products_clean,"olist_products_dataset")

olist_products_dataset : Saved


In [42]:
sellers = datasets["olist_sellers_dataset"]

In [43]:
reportSellers = data_quality_report(sellers, "seller_id")
print("Pour Table Sellers :")
print(f'{Affichage_Dic(reportSellers)}')

Pour Table Sellers :
- Rows: 3095
- Columns: 4
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0
None


In [44]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


In [45]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [46]:
sellers_clean = normalize_text_columns(
    sellers,
    ["seller_city", "seller_state"]
)

In [47]:
Affichage_Dic(data_quality_report(sellers_clean, "seller_id"))

- Rows: 3095
- Columns: 4
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [48]:
save_to_silver(sellers,"olist_sellers_dataset")

olist_sellers_dataset : Saved


In [49]:
order_items = datasets["olist_order_items_dataset"]

In [50]:
reportOrderItems = data_quality_report(order_items,["order_id","order_item_id"])
Affichage_Dic(reportOrderItems)

- Rows: 112650
- Columns: 7
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [51]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [52]:
order_items = convert_datetime_columns(order_items,["shipping_limit_date"])

In [53]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [54]:
order_items.describe()

,order_item_id,shipping_limit_date,price,freight_value
count,112650.000000,112650,112650.000000,112650.000000
mean,1.197834,2018-01-07 15:36:52.192685,120.653739,19.990320
min,1.000000,2016-09-19 00:15:34,0.850000,0.000000
25%,1.000000,2017-09-20 20:57:27.500000,39.900000,13.080000
50%,1.000000,2018-01-26 13:59:35,74.990000,16.260000
75%,1.000000,2018-05-10 14:34:00.750000,134.900000,21.150000
max,21.000000,2020-04-09 22:35:08,6735.000000,409.680000
std,0.705124,NaN,183.633928,15.806405


In [55]:
Affichage_Dic(data_quality_report(order_items, ["order_id","order_item_id"]))

- Rows: 112650
- Columns: 7
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [56]:
save_to_silver(order_items,"olist_order_items_dataset")

olist_order_items_dataset : Saved


In [57]:
order_payments = datasets["olist_order_payments_dataset"]

In [58]:
reportOrderPayments = data_quality_report(order_payments,["order_id", "payment_sequential"])
print("Pour Table Order Payments :")
Affichage_Dic(reportOrderPayments)

Pour Table Order Payments :
- Rows: 103886
- Columns: 5
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [59]:
order_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [60]:
order_payments.describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


In [61]:
# Vérifier les paiements avec 0 échéance.
# Une valeur payment_installments = 0 peut être incohérente si un montant a réellement été payé.
# On affiche payment_type et payment_value pour comprendre ces cas avant de décider du nettoyage.
order_payments[order_payments["payment_installments"] == 0][["payment_type", "payment_installments", "payment_value"]]

,payment_type,payment_installments,payment_value
46982,credit_card,0,58.69
79014,credit_card,0,129.94


In [62]:
# Vérifier les paiements dont le montant est égal à 0.
# L'objectif est de savoir si payment_value = 0 est associé à payment_installments = 0
order_payments[order_payments["payment_value"] == 0][["payment_type", "payment_installments", "payment_value"]]

,payment_type,payment_installments,payment_value
19922,voucher,1,0.0
36822,voucher,1,0.0
43744,voucher,1,0.0
51280,not_defined,1,0.0
57411,not_defined,1,0.0
62674,voucher,1,0.0
77885,voucher,1,0.0
94427,not_defined,1,0.0
100766,voucher,1,0.0


In [63]:
# Analyser la relation entre le nombre d'échéances et le montant du paiement.
# On calcule le montant moyen payé pour chaque nombre d'échéances afin de voir
# si les paiements plus élevés ont tendance à être répartis sur davantage d'échéances.
order_payments.groupby("payment_installments")["payment_value"].mean()

payment_installments
0      94.315000
1     112.420229
2     127.228150
3     142.539317
4     163.976840
5     183.465222
6     209.849952
7     187.673672
8     307.737427
9     203.440870
10    415.085837
11    124.932174
12    321.678496
13    150.462500
14    167.962667
15    445.553108
16    292.694000
17    174.602500
18    486.483333
20    615.801765
21    243.700000
22    228.710000
23    236.480000
24    610.048889
Name: payment_value, dtype: float64

In [64]:
order_payments_clean = clean_order_payments(order_payments)


In [65]:
order_payments_clean[(order_payments_clean["payment_installments"] == 0) & (order_payments_clean["payment_value"] > 0)
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [66]:
order_payments_clean = normalize_text_columns(
    order_payments_clean,
    ["payment_type"]
)

In [67]:
Affichage_Dic(data_quality_report(order_payments_clean, ["order_id", "payment_sequential"]))

- Rows: 103886
- Columns: 5
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [68]:
save_to_silver(order_payments_clean,"olist_order_payments_dataset")

olist_order_payments_dataset : Saved


In [69]:
order_reviews = datasets["olist_order_reviews_dataset"]

In [70]:
# review_id alone is not unique in this dataset because the same review_id
# can be associated with different orders.
# Therefore, (review_id, order_id) is used as a composite key
# for data quality checks.
reportOrderReviews = data_quality_report(order_reviews, ["review_id","order_id"])
print("Pour Table Order Reviews :")
Affichage_Dic(reportOrderReviews)

Pour Table Order Reviews :
- Rows: 99224
- Columns: 7
- Missing Values Count: 145903
- Duplicate Rows: 0
- Duplicate Keys: 0


In [71]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [72]:
order_reviews.describe()

,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [73]:
order_reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [74]:
order_reviews_clean = clean_order_reviews(order_reviews)

In [75]:
#Verifier la différence après le nettoyage
print(f"Avant : {order_reviews['review_comment_message'].isna().sum()}")
print(f"Après : {order_reviews_clean['review_comment_message'].isna().sum()}")

Avant : 58247
Après : 0


In [76]:
# Transformer les colonnes de dates en datetime
order_reviews_date_columns = ["review_creation_date", "review_answer_timestamp"]
order_review_clean = convert_datetime_columns(order_reviews_clean, order_reviews_date_columns)

In [77]:
order_reviews_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_comment_title     99224 non-null  str           
 4   review_comment_message   99224 non-null  str           
 5   review_creation_date     99224 non-null  datetime64[us]
 6   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 5.3 MB


In [78]:
Affichage_Dic(data_quality_report(order_reviews_clean, ["review_id","order_id"]))

- Rows: 99224
- Columns: 7
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [79]:
save_to_silver(order_review_clean,"olist_order_reviews_dataset")

olist_order_reviews_dataset : Saved


In [80]:
geolocation = datasets["olist_geolocation_dataset"]

In [81]:
reportGeolocation = data_quality_report(geolocation, None)
print("Pour Table Geolocation :")
Affichage_Dic(reportGeolocation)

Pour Table Geolocation :
- Rows: 1000163
- Columns: 5
- Missing Values Count: 0
- Duplicate Rows: 261831
- Duplicate Keys: None


In [82]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


In [83]:
geolocation[geolocation.duplicated()]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
...,...,...,...,...,...
1000153,99970,-28.343273,-51.873734,ciriaco,RS
1000154,99950,-28.070493,-52.011342,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS


In [84]:
geolocation_clean = normalize_text_columns(
    geolocation,
    ["geolocation_city", "geolocation_state"]
)

In [85]:
Affichage_Dic(data_quality_report(geolocation_clean, None))

- Rows: 1000163
- Columns: 5
- Missing Values Count: 0
- Duplicate Rows: 279667
- Duplicate Keys: None


In [88]:
geolocation_clean = clean_geolocation(geolocation_clean)

In [91]:
Affichage_Dic(data_quality_report(geolocation_clean, None))

- Rows: 720496
- Columns: 5
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: None


In [93]:
save_to_silver(geolocation_clean,"olist_geolocation_dataset")

olist_geolocation_dataset : Saved


In [94]:
translation = datasets["product_category_name_translation"]

In [95]:
reportTranslation = data_quality_report(translation, "product_category_name")
print("Pour Table Product Category Name Translation :")
Affichage_Dic(reportTranslation)

Pour Table Product Category Name Translation :
- Rows: 71
- Columns: 2
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [98]:
translation.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


In [101]:
translation_clean = normalize_text_columns( translation, ["product_category_name","product_category_name_english"] )

In [103]:
Affichage_Dic(data_quality_report(translation_clean,"product_category_name"))

- Rows: 71
- Columns: 2
- Missing Values Count: 0
- Duplicate Rows: 0
- Duplicate Keys: 0


In [104]:
save_to_silver(translation_clean,"product_category_name_translation")

product_category_name_translation : Saved
